# Data cleaning and normalizing scripts

## IEC-62443

In [ ]:
import pandas as pd
import json

# Definition of file paths based on the repository structure
source_files = [
    "../data/input/iec62443/SLES12-DISA-STIG.csv", 
    "../data/input/iec62443/SLES15-DISA-STIG.csv", 
    "../data/input/iec62443/SLES15-PCI-DSS.csv"    
]

def generate_clean_iec62443_dataset():
    """
    Reads SUSE files, extracts IEC 62443 standards and descriptions,
    normalizes SR tags, and exports a consolidated dictionary to JSON.
    """
    processed_dataframes = []
    
    # Function to handle poorly formatted tags (e.g., 'SR,1.1,SR,1.2' -> ['SR 1.1', 'SR 1.2'])
    def parse_sr_tags(raw_value):
        if pd.isna(raw_value) or str(raw_value).strip() == "":
            return []
        parts = str(raw_value).split(',')
        # Groups tokens into pairs to form the complete tag (e.g., 'SR' + '1.1')
        return [' '.join(parts[i:i+2]) for i in range(0, len(parts), 2)]

    for file_path in source_files:
        try:
            # Reading with ';' delimiter and windows-1252 encoding based on metadata
            temp_df = pd.read_csv(file_path, sep=';', encoding='windows-1252')
            
            # Extracting the standard (IEC.62443) and the technical description ('Name' column)
            # Note: 'Name' contains the technical rationale of the configuration, ideal for mapping
            processed_dataframes.append(temp_df[["IEC.62443", "Name"]])
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # 1. Union and Initial Cleaning
    consolidated_df = pd.concat(processed_dataframes, ignore_index=True)
    consolidated_df = consolidated_df.dropna(subset=["IEC.62443"])
    
    # 2. Standards Normalization (Explode)
    # Applying split to transform strings into lists and 'exploding' the dataframe
    consolidated_df["IEC.62443"] = consolidated_df["IEC.62443"].apply(parse_sr_tags)
    normalized_df = consolidated_df.explode("IEC.62443")
    
    # Removing empty entries resulting from whitespaces or nulls and exact duplicates
    normalized_df = normalized_df[normalized_df["IEC.62443"] != ""].drop_duplicates()
    
    # 3. Semantic Aggregation
    # For each unique standard, group all descriptions into a single block of text.
    # This creates a denser semantic anchor for the embeddings model.
    final_dictionary = (
        normalized_df.groupby("IEC.62443")["Name"]
        .unique()
        .apply(lambda x: " ".join(x))
        .to_dict()
    )
    
    # 4. JSON Export
    output_file_name = "../data/output/datasets/iec62443.json"
    with open(output_file_name, "w", encoding="utf-8") as f:
        json.dump(final_dictionary, f, indent=4, ensure_ascii=False)
    
    print(f"Processing completed: {len(final_dictionary)} unique requirements mapped to '{output_file_name}'.")

# Execute function within the notebook environment
generate_clean_iec62443_dataset()

Processamento concluído: 42 requisitos únicos mapeados em '../data/output/iec62443.json'.


## CWE

In [ ]:
import xml.etree.ElementTree as ET
import json
import re

def generate_clean_cwe_dataset():
    """
    Reads the official MITRE XML, extracts weakness IDs and descriptions,
    and exports a flattened JSON optimized for NLP models.
    """
    print("Loading and parsing XML. This may take a few seconds...")
    tree = ET.parse('../data/input/cwe/cwec_v4.19.1.xml')
    root = tree.getroot()
    
    # The CWE XML uses Namespaces (e.g., xmlns="http://cwe.mitre.org/cwe-7").
    # We need to capture this namespace dynamically for the search to function.
    ns = {'cwe': root.tag.split('}')[0].strip('{')} if '}' in root.tag else {}
    
    # Defining search paths considering the namespace
    weaknesses_search = './/cwe:Weaknesses/cwe:Weakness' if ns else './/Weaknesses/Weakness'
    desc_search = 'cwe:Description' if ns else 'Description'
    ext_desc_search = 'cwe:Extended_Description' if ns else 'Extended_Description'

    cwe_dictionary = {}

    # Helper function to extract all text, even if there are nested HTML/XML tags
    def extract_clean_text(element):
        if element is None:
            return ""
        # itertext() fetches all text ignoring inner tags (e.g., <b>, <p>)
        text = " ".join(element.itertext())
        # Cleans line breaks and multiple spaces
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    for weakness in root.findall(weaknesses_search, ns):
        cwe_id = weakness.attrib.get('ID')
        name = weakness.attrib.get('Name', '')
        
        # Extracts the short description and the extended description (rich in security context)
        desc_elem = weakness.find(desc_search, ns)
        description = extract_clean_text(desc_elem)
        
        ext_desc_elem = weakness.find(ext_desc_search, ns)
        ext_desc = extract_clean_text(ext_desc_elem)
        
        if cwe_id:
            cwe_tag = f"CWE-{cwe_id}"
            
            # Concatenates fields to create the requirement's "semantic anchor"
            full_text = f"{name}. {description} {ext_desc}".strip()
            cwe_dictionary[cwe_tag] = full_text

    # Export
    output_file_name = "../data/output/datasets/cwe.json"
    with open(output_file_name, 'w', encoding='utf-8') as f:
        json.dump(cwe_dictionary, f, indent=4, ensure_ascii=False)
        
    print(f"Processing completed: {len(cwe_dictionary)} CWEs mapped and saved to '{output_file_name}'.")

# Executing the function with the attached file
generate_clean_cwe_dataset()

Carregando e fazendo o parse do XML. Isso pode levar alguns segundos...
Processamento concluído: 969 CWEs mapeados e salvos em '../data/output/cwe.json'.


## CIS Benchmark

In [ ]:
import requests
import yaml
import json
import re

def extract_cis_docker_from_aqua(output_file):
    print("🌐 Downloading the official CIS Docker Benchmark from Aqua Security...")
    
    # Direct URL to the raw YAML file on their GitHub (CIS version 1.6.0)
    yaml_url = "https://raw.githubusercontent.com/aquasecurity/docker-bench/main/cfg/cis-1.6.0/definitions.yaml"
    
    try:
        response = requests.get(yaml_url)
        response.raise_for_status()
        # BaseLoader prevents YAML from breaking with strange characters
        cis_data = yaml.load(response.text, Loader=yaml.BaseLoader)
    except Exception as e:
        print(f"❌ Error downloading file: {e}")
        return

    cis_docker_dataset = {}
    total_rules = 0

    print("🧩 Processing groups and extracting rules...")
    
    # The YAML is divided into "groups" (CIS Sections)
    groups = cis_data.get('groups', [])
    for group in groups:
        group_id = str(group.get('id', ''))
        
        # We want ONLY Section 4 (Container Images and Build File)
        # To extract the entire manual, simply remove the 'if' statement below.
        if not group_id.startswith('4'):
            continue
            
        checks = group.get('checks', [])
        for check in checks:
            check_id = str(check.get('id', ''))
            description = check.get('description', '').strip()
            remediation = check.get('remediation', '').strip()
            
            # Cleans the remediation text (removes line breaks and pipes)
            clean_reremediation = re.sub(r'\s+', ' ', remediation).strip()
            clean_reremediation = clean_reremediation.replace('|', '')
            
            # Removes the "(Scored) / (Not Scored)" tag from the title to keep it clean for the AI
            clean_description = re.sub(r'\s*\(Not Scored\)|\s*\(Scored\)', '', description, flags=re.IGNORECASE)
            
            # Concatenates to create the dense NLP block
            dense_text = f"{clean_description}. {clean_reremediation}".strip()
            
            if dense_text:
                # User-friendly key, e.g., "CIS_4.1"
                nlp_key = f"CIS_{check_id}"
                cis_docker_dataset[nlp_key] = dense_text
                total_rules += 1

    # Exporting to the Destination Dataset
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cis_docker_dataset, f, indent=4, ensure_ascii=False)

    print("="*40)
    print("🎯 CIS DOCKER DATASET REPORT")
    print("="*40)
    print(f"Section 4 rules extracted: {total_rules}")
    print(f"File successfully saved to:  {output_file}")
    print("\n✅ Part B (Destination) 100% Completed!")

# Run the extractor
extract_cis_docker_from_aqua('../data/output/datasets/cis_docker_benchmark.json')

🌐 Baixando o CIS Docker Benchmark oficial da Aqua Security...
🧩 Processando os grupos e extraindo as regras...
🎯 RELATÓRIO DO DATASET CIS DOCKER
Regras da Seção 4 extraídas: 12
Arquivo salvo com sucesso em:  ../data/output/cis_docker_benchmark.json

✅ Metade B (Destino) 100% Finalizada!


## Hadolint e ShellCheck

In [ ]:
import os
import subprocess
import json
import re

def clean_wiki_text_for_nlp(md_content):
    # 1. Removes code blocks (Pure syntax does not help the AI)
    text = re.sub(r'```.*?```', '', md_content, flags=re.DOTALL)
    
    # 2. Removes "useless" subheadings for the model (Boilerplate)
    text = re.sub(r'(?i)#*\s*(?:Problematic code|Correct code|Rationale|Exceptions)[:]*', '', text)
    
    # 3. Removes only the rule tag at the top (e.g., "# DL4003") to keep the vocabulary clean
    text = re.sub(r'^#+\s*(?:DL|SC)\d+\s*$', '', text, flags=re.MULTILINE|re.IGNORECASE)
    
    # 4. Removes only the remaining '#' SYMBOLS, preserving the rich text next to them
    text = re.sub(r'#+', '', text)
    
    # 5. Cleans Markdown links syntax [text](url), keeping the text
    text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text)
    
    # 6. Removes loose URLs and prefixes ("See also:", "Reference:", etc.)
    text = re.sub(r'(?i)(?:see(?:\s+also)?|read(?:\s+more)?|reference[s]?|for\s+more\s+(?:info|details))?[\s:]*http[s]?://\S+', '', text)
    
    # 7. Removes heavy Markdown formatting (*, _, `, >, etc)
    text = re.sub(r'[*`_~>|-]', '', text)
    
    # 8. Flattens the text to generate a dense and continuous block
    dense_text = re.sub(r'\s+', ' ', text).strip()
    
    return dense_text

def clone_and_extract_wikis(output_file):
    source_dataset = {}
    total_dl = 0
    total_sc = 0

    print("="*55)
    print("🚀 STARTING STANDARDIZED INGESTION (LOCAL WIKIS)")
    print("="*55)

    # Configuration dictionary for Wikis
    wiki_repositories = {
        "hadolint": {
            "url": "https://github.com/hadolint/hadolint.wiki.git",
            "pasta": "../data/input/hadolint/wiki_hadolint_temp",
            "regex": r'^DL\d{4}\.md$'
        },
        "shellcheck": {
            "url": "https://github.com/koalaman/shellcheck.wiki.git",
            "pasta": "../data/input/hadolint/wiki_shellcheck_temp",
            "regex": r'^SC\d{4}\.md$'
        }
    }

    # ---------------------------------------------------------
    # CLONING AND EXTRACTION
    # ---------------------------------------------------------
    for project, config in wiki_repositories.items():
        print(f"\n📚 Processing {project.upper()} Wiki...")
        folder = config["pasta"]
        
        # Performs Git Clone only if the folder does not exist yet
        if not os.path.exists(folder):
            print(f"   📥 Cloning repository: {config['url']}")
            try:
                subprocess.run(["git", "clone", config["url"], folder], check=True, capture_output=True)
            except subprocess.CalledProcessError as e:
                print(f"   ❌ Error cloning {project} wiki. Git returned an error.")
                continue
        else:
            print("   (Repository already found locally. Reading files...)")

        # Scans the Wiki folder looking for the target files
        for file_name in os.listdir(folder):
            if re.match(config["regex"], file_name):
                full_path = os.path.join(folder, file_name)
                
                try:
                    with open(full_path, 'r', encoding='utf-8') as f:
                        clean_text = clean_wiki_text_for_nlp(f.read())
                        
                    if clean_text:
                        rule_id = file_name.replace('.md', '')
                        source_dataset[rule_id] = clean_text
                        
                        if project == "hadolint":
                            total_dl += 1
                        else:
                            total_sc += 1
                except Exception as e:
                    pass # Silently ignore if there is a reading issue with a specific file

    # ---------------------------------------------------------
    # EXPORT
    # ---------------------------------------------------------
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(source_dataset, f, indent=4, ensure_ascii=False)

    print("\n" + "="*55)
    print("🎯 MASTER SOURCE DATASET REPORT")
    print("="*55)
    print(f"Hadolint rules (DL) extracted:   {total_dl}")
    print(f"ShellCheck rules (SC) extracted: {total_sc}")
    print(f"Total vectors for AI:            {total_dl + total_sc}")
    print(f"Generated file:                  {output_file}")
    print("\n✅ Consistent Source Base 100% Completed!")

# Ready to run! It will create the wiki_hadolint_temp and wiki_shellcheck_temp folders in your directory.
clone_and_extract_wikis('../data/output/datasets/hadolint_rules.json')

🚀 INICIANDO INGESTÃO PADRONIZADA (WIKIS LOCAIS)

📚 Processando a Wiki do HADOLINT...
   📥 Clonando repositório: https://github.com/hadolint/hadolint.wiki.git

📚 Processando a Wiki do SHELLCHECK...
   📥 Clonando repositório: https://github.com/koalaman/shellcheck.wiki.git

🎯 RELATÓRIO DO DATASET MESTRE DE ORIGEM
Regras Hadolint (DL) extraídas:   69
Regras ShellCheck (SC) extraídas: 520
Total de vetores para a IA:       589
Arquivo gerado:                   ../data/output/hadolint_rules.json

✅ Base de Origem Consistente e 100% Finalizada!
